In [ ]:
import ray
from ray.train import ScalingConfig, RunConfig
from ray.train.torch import TorchTrainer
from ray import tune
from ray.tune import Tuner, TuneConfig
from ray import serve
import torch
import requests
from requests import Request
import pandas as pd
import json
from intro import clean_and_combine, VectorizeAndEncode, ProductClassifier, train_loop, train_driver

# Introduction - Ray - Anyscale - AI Libraries - Ray Core

## What is Ray?

<img src='https://docs.ray.io/en/releases-2.38.0/_images/map-of-ray.svg' width=700 />

## Ray

* OSS framework for high-performance, resilient, scale-out computation on heterogeneous hardware
* Distributed scheduler supporting stateless functions ("tasks") as well as long-running stateful processes ("actors")
* Key features: 
  * Dependency tracking (task graphs)
  * Data movement and resource aware
  * Supports mix of resource requirements (e.g., GPUs), fractional, and custom resources
* Additional infra: object store, fault tolerance via GCS
* Ray AI Libraries are a set of high-level APIs for accomplishing common large-scale data + compute use cases (e.g., data transformation, model training)
* Easy, Python-based APIs and coding patterns

## Anyscale: Production-ready Ray from day one

* __Developer central__: multi-node backed IDE, advanced observability by Ray library
* __Optimized runtime__: faster performance and higher GPU utilization vs. OSS
* __Cluster controller__: proactive unhealthy node replacement, 0-100 node 60-sec cold starts
* __Expertise__: Training, 24/7 support, professional services


# Overview of the Ray AI Libraries

Built on top of Ray Core, the Ray AI Libraries inherit all the performance and scalability benefits offered by Core while providing a convenient abstraction layer for machine learning. These Python-first native libraries allow ML practitioners to distribute individual workloads, end-to-end applications, and build custom use cases in a unified framework.

The Ray AI Libraries bring together an ever-growing ecosystem of integrations with popular machine learning frameworks to create a common interface for development.

|<img src="https://technical-training-assets.s3.us-west-2.amazonaws.com/Introduction_to_Ray_AIR/e2e_air.png" width="100%" loading="lazy">|
|:-:|
|Ray AI Libraries enable end-to-end ML development and provides multiple options for integrating with other tools and libraries form the MLOps ecosystem.|



# End-to-End Demo: Product Category Prediction 

## Load and process data with Ray Data

In [ ]:
ds = ray.data.read_csv("s3://anyscale-public-materials-use2/ecom/intro/ecommerce_product_catalog.csv")

pd.DataFrame(ds.take(5))

### Clean and combine text fields

We lowercase, strip punctuation, and concatenate `title` and `description` into a single `text` column. This is a **per-row** transform, so we use `ds.map()`.

In [ ]:
ds_cleaned = ds.map(clean_and_combine)

pd.DataFrame(ds_cleaned.take(5))

### Vectorize and encode with `map_batches`

Now we apply the fitted vectorizer and the label mapping in a single `map_batches` call. This is a **stateful, batch transform**. Ray Data will run this in parallel across its block partitions.

In [ ]:
ds_encoded = ds_cleaned.map_batches(VectorizeAndEncode)

pd.DataFrame(ds_encoded.take(5))

### Train / test split

In [ ]:
train_ds, val_ds = ds_encoded.train_test_split(test_size=0.2, seed=42)

print(f"Train: {train_ds.count()} rows")
print(f"Val:   {val_ds.count()} rows")

## Train a PyTorch classifier with Ray Train

We define a small feedforward network and train it using `TorchTrainer`.

In [ ]:
trainer = TorchTrainer(
    train_loop_per_worker=train_loop,
    scaling_config=ScalingConfig(num_workers=2),
    datasets={"train": train_ds},
    run_config=RunConfig(storage_path='/mnt/cluster_storage/'),
    train_loop_config={"lr" : 1e-2}
)

result = trainer.fit()

result.metrics_dataframe

## Optimize hyperparameters with Ray Tune

Tune allows us to maximize cluster utilization by running multiple experiments, each of which may require multiple workers/GPUs.

The example below features simple random search, buy by using built-in integrations for efficient schedulers and search algorithms, we can immediately use knowledge from completed trials to schedule additional trials.

In [ ]:
tuner = Tuner(
    tune.with_parameters(train_driver, dataset=train_ds),
    param_space={        
        "lr": tune.loguniform(1e-4, 1e-1), # example: value Tune actually searches over
    },
    tune_config=TuneConfig(
        metric="loss",
        mode="min",
        num_samples=3,        
    ),
)

tune_results = tuner.fit()

In [ ]:
tune_results.get_dataframe()

In [ ]:
best_result_path = tune_results.get_best_result("loss", mode="min").metrics['checkpoint_path']

best_result_path

## Batch inference with Ray Data

Batch inference is implemented similar to feature engineering using stateful computation. In the case of inference, the state is the model we want to load and re-use for many batches of data.

In [ ]:
class OfflinePredictor:
    def __init__(self):
        # Load expensive state
        self._model = ProductClassifier()
        self._model.load_state_dict(torch.load(best_result_path + '/model.pt', weights_only=True))

    def __call__(self, batch: dict) -> dict:
        # Make prediction in batch
        with torch.inference_mode():
            outputs = self._model(torch.tensor(batch['features']))
        return {"prediction": outputs.numpy()}

In [ ]:
predictions = val_ds.select_columns(['features']).map_batches(OfflinePredictor, concurrency=2)
predictions.take_batch(3)

## Online prediction with Ray Serve

For low-latency, high-performance serving, we code and enhance a simple Python class to create a `Deployment`. Deployments can be composed to allow easy integration while retaining good separation-of-concerns patterns, effective development, and simple upgrades.

In [ ]:
@serve.deployment
class OnlinePredictor:
    def __init__(self, checkpoint):
        self._model = ProductClassifier()
        self._model.load_state_dict(torch.load(checkpoint, weights_only=True))

    async def __call__(self, request: Request) -> dict:
        data = await request.json()
        return {"prediction": self.predict(data)}

    def predict(self, data: dict) -> list[float]:
        with torch.inference_mode():
            outputs = self._model(torch.tensor(data["features"]))
        return outputs.numpy().tolist()

handle = serve.run(OnlinePredictor.bind(checkpoint=best_result_path + '/model.pt'))

In [ ]:
# Form payload
sample_inputs = val_ds.select_columns(['features'])
sample = sample_inputs.take(1)[0]['features'].astype('float64')

# Send HTTP request
requests.post("http://localhost:8000/", json={'features' : list(sample)}).json()

In [ ]:
# Shutdown Ray Serve
serve.shutdown()

# A Brief Look at Ray Core

## Ray Core overview

Ray Core is about:
* distributing computation across many cores, nodes, or devices (e.g., accelerators)
* scheduling *arbitrary task graphs*
    * any code you can write, you can distribute, scale, and accelerate with Ray Core
* manage the overhead
    * at scale, distributed computation introduces growing "frictions" -- data movement, scheduling costs, etc. -- which make the problem harder
    * Ray Core addresses these issues as first-order concerns in its design (e.g., via a distributed scheduler)
 
(And, of course, for common technical use cases, libraries and other components provide simple dev ex and are built on top of Ray Core)

## `@ray.remote` and `ray.get`

Here is a diagram which shows the relationship between Python code and Ray tasks.

<img src="https://technical-training-assets.s3.us-west-2.amazonaws.com/Ray_Core/python_to_ray_task_map.png" width="80%" >

Define a Python function and decorate it so that Ray can schedule it

In [ ]:
@ray.remote(num_cpus=2)
def f(a, b):
    return a + b

Tell Ray to schedule the function

In [ ]:
f.remote(1, 2)

`ObjectRef` is a handle to a task result. We get an ObjectRef immediately because we don't know
* when the task will run
* whether it will succeed
* whether we really need or want the result locally
    * consider a very large result which we may need for other work but which we don't need to inspect

In [ ]:
ref = f.remote(1, 2)

If we want to wait (block) and retrieve the corresponding object, we can use `ray.get`

In [ ]:
ray.get(ref)

### Task graphs

The above example is a common scenario, but it is also the easiest (least complex) scheduling scenario. Each task is independent of the others -- this is called "embarrassingly parallel"

Many real-world algorithms are not embarrassingly parallel: some tasks depend on results from one or more other tasks. Scheduling this graphs is more challenging.

Ray Core is designed to make this straightforward

In [ ]:
@ray.remote
def add(a, b):
    return a+b

In [ ]:
arg1 = add.remote(1,2)

arg1

In [ ]:
arg2 = add.remote(10, 20)

We want to schedule `add` which depends on two prior invocations of `add`

We can pass the resulting ObjectRefs -- this means 
* we don't have to wait for the dependencies to complete before we can set up `add` for scheduling
* we don't need to have the concrete parameters (Python objects) for the call to `add.remote`
* Ray will automatically resolve the ObjectRefs -- our `add` implementation will never know that we passed ObjectRefs, not, e.g., numbers

In [ ]:
out = add.remote(arg1, arg2)

In [ ]:
ray.get(out)

## Ray Actors

Actors are Python class instances which can run for a long time in the cluster, which can maintain state, and which can send messages to/from other code.

Let's look at an example of an actor which maintains a running balance.

In [ ]:
@ray.remote
class Accounting:
    def __init__(self):
        self._total = 0
    
    def add(self, amount):
        self._total += amount
        
    def remove(self, amount):
        self._total -= amount
        
    def total(self):
        return self._total

<div class="alert alert-block alert-warning">

<b>Note:</b> The most common use case for actors is with state that is not mutated but is large enough that we may want to load it only once and ensure we can route calls to it over time, such as a large AI model.

</div>

Define an actor with the `@ray.remote` decorator and then use `<class_name>.remote()` ask Ray to construct and instance of this actor somewhere in the cluster.

We get an actor handle which we can use to communicate with that actor, pass to other code, tasks, or actors, etc.

In [ ]:
acc = Accounting.remote()

We can send a message to an actor -- with RPC semantics -- by using `<handle>.<method_name>.remote()`

In [ ]:
acc.total.remote()

Not surprisingly, we get an object ref back

In [ ]:
ray.get(acc.total.remote())

We can mutate the state inside this actor instance

In [ ]:
acc.add.remote(100)

In [ ]:
acc.remove.remote(10)

In [ ]:
ray.get(acc.total.remote())